# Task 3 — Aula Abierta (Grupo 4)

Notebook del laboratorio: sensibilidad a `H`, comparación con el baseline generado por IA y 12 políticas de autoescalado.

| Parámetro | Valor |
|-----------|-------|
| Grupo | 4 |
| `q` | 0.35 /día |
| `D` | 6 días |
| Semilla base | 20261022 |
| Réplicas | 20 por escenario |

**Motor:** `Task_3/src/hybrid_engine.py` (extraído de Task 2, sin modificar Tasks 1–2).

Por defecto **carga resultados ya calculados** en `resultados/`. Pon `FORCE = True` en la celda de configuración si quieres recalcular (tarda varios minutos).

## 0. Configuración e imports

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Rutas: funciona desde Task_3/ o desde la raíz del repo
ROOT = Path.cwd()
if ROOT.name != 'Task_3':
    if (ROOT / 'Task_3').is_dir():
        ROOT = ROOT / 'Task_3'
    else:
        raise RuntimeError('Abre el notebook desde Task_3/ o desde la raíz del repo Lab6-MS')

SRC = ROOT / 'src'
RES = ROOT / 'resultados'
FIG = ROOT / 'figuras'
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(ROOT))

from hybrid_engine import (
    BASE_SEED, ci, hybrid, net_income, rrmse, smoke_match_task2, wilson_ci,
)
from baseline_runner import correr_hibrido as baseline_run
import task_3 as t3

# --- controles ---
FORCE = False          # True = recalcular aunque existan CSV (tarda varios minutos)
SHOW_FIGS = True       # mostrar PNG guardados

print('ROOT =', ROOT)
print('Semilla base =', BASE_SEED)
print('FORCE =', FORCE)

## 1. Validación frente a Task 2

In [ ]:
smoke = smoke_match_task2(BASE_SEED)
display(Markdown(f"**Smoke vs Task 2:** `ok = {smoke.get('ok')}`"))
display(pd.DataFrame([
    {'modelo': 'Task_3 (nuestro)', **smoke['ours']},
    {'modelo': 'Task_2', **(smoke['task2'] or {})},
]))
assert smoke.get('ok'), smoke
print('Motor híbrido coincide con Task 2 en first_order, U(100) y c(100).')

## 2. Task 3.1 — Sensibilidad a H

Horizonte 100 días, `H ∈ {0.25, 0.5, 1, 2, 4, 7}`, referencia emparejada `H=0.25`.  
RRMSE de `U(t)` en malla diaria `t = 0…100`.

In [ ]:
rep_path = RES / 'sensibilidad_H_replicas.csv'
sum_path = RES / 'sensibilidad_H_resumen.csv'

if FORCE or not rep_path.exists():
    print('Ejecutando Task 3.1 (puede tardar ~2 min con 20 réplicas)...')
    # usa la implementación completa del script
    out31 = t3.run_task_31(force=True)
else:
    print('Cargando resultados existentes de 3.1')
    out31 = {
        'rows': t3.read_csv(rep_path),
        'summary': t3.read_csv(sum_path),
        'meta': json.loads((RES / 'sensibilidad_H_meta.json').read_text()),
    }

df31 = pd.DataFrame(out31['rows'] if 'rows' in out31 else t3.read_csv(rep_path))
for col in ['H', 'U100', 'Wmax', 'rmax', 'RRMSE', 'RRMSE_pct']:
    df31[col] = pd.to_numeric(df31[col], errors='coerce')

# tabla resumen pivoteada
sum_rows = t3.read_csv(sum_path)
sum_df = pd.DataFrame(sum_rows)
sum_df[['H', 'mean', 'low', 'high']] = sum_df[['H', 'mean', 'low', 'high']].apply(pd.to_numeric)

def fmt_ci(metric):
    sub = sum_df[sum_df['metric'] == metric].sort_values('H')
    return pd.DataFrame({
        'H': sub['H'].values,
        'media': sub['mean'].values,
        'IC_low': sub['low'].values,
        'IC_high': sub['high'].values,
    })

display(Markdown('### RRMSE (%) por H'))
display(fmt_ci('RRMSE_pct').round(3))
display(Markdown('### U(100) por H'))
display(fmt_ci('U100').round(2))
display(Markdown('### W_max (s) por H'))
display(fmt_ci('Wmax').round(4))

In [ ]:
# Criterio de confianza: primer H con RRMSE medio > 5 %
rr = fmt_ci('RRMSE_pct')
first_fail = None
for _, row in rr.iterrows():
    if row['H'] == 0.25:
        continue
    if row['media'] > 5:
        first_fail = row['H']
        break

display(Markdown(
    f"**Criterio declarado (no docente):** RRMSE medio > 5 %. "
    f"Primer H que incumple: **{first_fail}**."
))

if SHOW_FIGS:
    for name in ['error_vs_H.png', 'usuarios_vs_t_H.png', 'latencia_vs_H.png']:
        p = FIG / name
        if p.exists():
            display(Markdown(f'#### `{name}`'))
            display(Image(filename=str(p)))

## 3. Task 3.2 — Baseline IA vs modelo verificado

Escenario comparable: H=1, 100 días, θ=0.7, k=2, D=6.

In [ ]:
delta_path = RES / 'delta_baseline.csv'
if FORCE or not delta_path.exists():
    print('Ejecutando Task 3.2...')
    out32 = t3.run_task_32(force=True)
else:
    print('Cargando resultados existentes de 3.2')
    out32 = json.loads((RES / 'delta_baseline.json').read_text())

df32 = pd.read_csv(delta_path)
display(Markdown('### Medias e IC 95 %'))
rows = []
for model in ['baseline', 'verificado']:
    sub = df32[df32['model'] == model]
    for m in ['U100', 'c100', 'Wmax', 'rmax', 'first_order', 'first_delivery', 'n_orders']:
        vals = pd.to_numeric(sub[m], errors='coerce').dropna().values
        s = ci(vals)
        rows.append({'modelo': model, 'métrica': m, 'media': s['mean'], 'IC_low': s['low'], 'IC_high': s['high']})
display(pd.DataFrame(rows).round(4))

display(Markdown(
    'Diferencia clave: el baseline pide en el día **14.0**; el verificado localiza el cruce en **≈13.72**.'
))

if SHOW_FIGS and (FIG / 'baseline_vs_verificado.png').exists():
    display(Image(filename=str(FIG / 'baseline_vs_verificado.png')))

## 4. Task 3.3 — Doce políticas `(θ, k)`

150 días, H=1, θ ∈ {0.5,0.6,0.7,0.8}, k ∈ {1,2,4}, 20 réplicas.  
Ingreso índice: `N = Σ U_d − 40 Σ c_d` (t = d).  
Excedencia: algún día con `r_d > 0.05`.

In [ ]:
pol_sum = RES / 'politicas_resumen.csv'
pol_rep = RES / 'politicas_replicas.csv'

if FORCE or not pol_sum.exists():
    print('Ejecutando Task 3.3 (≈5 min con 240 corridas)...')
    out33 = t3.run_task_33(force=True)
else:
    print('Cargando resultados existentes de 3.3')

df_pol = pd.read_csv(pol_sum)
df_pol = df_pol.sort_values('ingreso_medio', ascending=False)
display(Markdown('### Ranking por ingreso medio'))
cols = ['theta', 'k', 'ingreso_medio', 'ingreso_low', 'ingreso_high',
        'p_exceder5', 'p_low', 'p_high', 'U150', 'c150', 'Wmax', 'n_orders']
display(df_pol[cols].round(3))

best = df_pol.iloc[0]
risk_ok = df_pol[df_pol['p_high'] <= 0.25]
chosen = risk_ok.sort_values('ingreso_medio', ascending=False).iloc[0] if len(risk_ok) else best

display(Markdown(
    f"**Mejor ingreso:** θ={best['theta']}, k={int(best['k'])}, "
    f"N̄={best['ingreso_medio']:.1f}.\n\n"
    f"**Recomendación (Wilson superior ≤ 0.25, luego max ingreso):** "
    f"θ={chosen['theta']}, k={int(chosen['k'])}."
))

if SHOW_FIGS:
    for name in ['ingreso_por_politica.png', 'riesgo_por_politica.png', 'frontera_ingreso_riesgo.png']:
        p = FIG / name
        if p.exists():
            display(Markdown(f'#### `{name}`'))
            display(Image(filename=str(p)))

## 5. Comparaciones estadísticas (Holm)

In [ ]:
pairs_path = RES / 'politicas_pareadas.csv'
if pairs_path.exists():
    pairs = pd.read_csv(pairs_path)
    nsig = (pairs['sig_0.05'] == 0).sum()
    display(Markdown(
        f'De **{len(pairs)}** parejas de ingreso (t emparejada + Holm α=0.05), '
        f'**{nsig}** no alcanzan significancia (no implica equivalencia).'
    ))
    display(pairs.sort_values('p_holm').head(10).round(4))
else:
    print('No hay politicas_pareadas.csv; ejecuta Task 3.3 primero.')

## 6. Conclusiones (Task 3.4)

1. **H:** confiar en H ≤ 1; desde H = 2 el RRMSE medio supera el 5 %. H = 7 engaña en W_max por pocas ventanas.
2. **Baseline IA:** estructura correcta, pero atrasa el primer pedido (~0.28 días) al no localizar el cruce continuo del umbral.
3. **Política:** bajo el criterio ingreso–riesgo declarado, conviene **θ = 0.5, k = 2**.
4. **Datos a pedir a la empresa:** series horarias de solicitudes + usuarios, latencia, rechazos, abandono y capacidad, para recalibrar `a` y los κ.

Más detalle narrativo: `resumen_resultados.md` y `resumen_para_reporte.md`.

In [ ]:
# Regenerar markdowns del informe a partir de los CSV/JSON actuales
t3.write_docs()
t3.write_readme()
print('Documentos actualizados en', ROOT)

## Demo rápida (opcional)

Una sola réplica del híbrido exacto para inspeccionar trayectoria.

In [ ]:
run = hybrid(mode='exact', seed=BASE_SEED, H=1.0, horizon=100.0, theta=0.7, k_order=2)
print(f"U(100)={run['U_end']:.2f}  c(100)={run['c_end']}  "
      f"Wmax={run['Wmax']:.4f}  first_order={run['first_order']:.4f}")

fig, axs = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
tr = np.array(run['trajectory'])
axs[0].plot(tr[:, 0], tr[:, 1])
axs[0].set_ylabel('U(t)')
axs[1].step(run['grid_t'], run['W_grid'], where='post')
axs[1].axhline(0.75, color='gray', ls='--', label='W_ok')
axs[1].set_ylabel('W (s)')
axs[1].legend()
axs[2].step(run['grid_t'], run['c_grid'], where='post')
axs[2].set_ylabel('c')
axs[2].set_xlabel('t (días)')
fig.suptitle('Réplica demo H=1, θ=0.7, k=2')
fig.tight_layout()
demo_path = FIG / 'demo_hibrido.png'
FIG.mkdir(parents=True, exist_ok=True)
fig.savefig(demo_path, dpi=140)
plt.close(fig)
display(Image(filename=str(demo_path)))